In [52]:
from pymilvus import MilvusClient
from glob import glob

client = MilvusClient(uri="http://localhost:19530", db_name="default", timeout=10)

collection_name = "rag_docs"
if client.has_collection(collection_name=collection_name):
    client.drop_collection(collection_name=collection_name)
client.create_collection(collection_name=collection_name,dimension=1536)

In [4]:
# 处理数据
text_lines = []
for file_path in glob("milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path) as f:
        file_text = f.read()
    text_lines += file_text.split("# ")

In [8]:
print(len(text_lines))
print(text_lines[:3])

72
['---\nid: operational_faq.md\nsummary: Find answers to commonly asked questions about operations in Milvus.\ntitle: Operational FAQ\n---\n\n', 'Operational FAQ\n\n<!-- TOC -->\n\n\n<!-- /TOC -->\n\n###', 'What if I failed to pull the Milvus Docker image from Docker Hub?\n\nIf you failed to pull the Milvus Docker image from Docker Hub, try adding other registry mirrors. \n\nUsers from Mainland China can add the URL "https://registry.docker-cn.com" to the registry-mirrors array in **/etc.docker/daemon.json**.\n\n```\n{\n  "registry-mirrors": ["https://registry.docker-cn.com"]\n}\n```\n\n###']


In [9]:
#embedding
from openai import OpenAI
openai_client=OpenAI()

def emb_text(text):
    return openai_client.embeddings.create(input=text,model="text-embedding-3-small").data[0].embedding

In [23]:
test_emb=emb_text("This is a test")
print(test_emb)
print(len(test_emb))

[0.00988006591796875, -0.005588531494140625, 0.00682830810546875, -0.038055419921875, -0.018218994140625, -0.041259765625, -0.007656097412109375, 0.0322265625, 0.0189208984375, 7.772445678710938e-05, 0.058929443359375, 0.0124969482421875, -0.0233612060546875, -0.01910400390625, 0.0255279541015625, 0.029571533203125, -0.08135986328125, 0.0016841888427734375, -0.020050048828125, 0.0282745361328125, 0.0263214111328125, -0.0087127685546875, 0.03948974609375, 0.0029754638671875, 0.02734375, -0.042633056640625, -0.024322509765625, 0.002605438232421875, 0.022308349609375, -0.055389404296875, 0.046630859375, -0.03436279296875, -0.0306549072265625, -0.01287078857421875, -0.01654052734375, -0.00034928321838378906, -0.00600433349609375, 0.034576416015625, -0.0335693359375, 0.0007834434509277344, -0.03607177734375, -0.052520751953125, 0.00521087646484375, -0.004543304443359375, 0.018829345703125, 0.019989013671875, -0.034210205078125, -0.0203704833984375, 0.01372528076171875, 0.042572021484375, -0

In [18]:

vectors=[emb_text(each) for each in text_lines]
docs=[{"id":i,"vector":vectors[i],"text":text_lines[i]}for i in range(len(vectors))]

In [19]:
client.insert(collection_name=collection_name,data=docs)

{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

In [24]:
query="how is data stored in milvus"
search_res=client.search(
    collection_name=collection_name,
    data=[emb_text(query)],
    limit=2,
    output_fields=["text"]
)

In [28]:
print(search_res)
print(len(search_res))

data: [[{'id': 44, 'distance': 0.7631347179412842, 'entity': {'text': ' Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###'}}, {'id': 70, 'distance': 0.6559406518936157, 'entity': {'text': 'How does Milvus ha

In [29]:
import json

retrieved_lines_with_distances=[(res['entity']['text'],res['distance']) for res in search_res[0]]

In [30]:
print(retrieved_lines_with_distances)

[(' Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###', 0.7631347179412842), ('How does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float32, Float16, and BFloat16 vector types.\

In [37]:
import json
#将python对象转换为json格式的字符串
print(json.dumps(retrieved_lines_with_distances,indent=4))

[
    [
        " Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.7631347179412842
    ],
    [
        "How does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float

In [41]:
#使用LLM获取RAG响应
content=""
for line in retrieved_lines_with_distances:
    content+='参考资料:'+line[0]+'----------------------------\n'
print(content)


参考资料: Where does Milvus store data?

Milvus deals with two types of data, inserted data and metadata. 

Inserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).

Metadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.

###----------------------------
参考资料:How does Milvus handle vector data types and precision?

Milvus supports Binary, Float32, Float16, and BFloat16 vector types.


In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_core.prompts import PromptTemplate
model=ChatTongyi(name="qwen3-max") # type: ignore
prompt_template=PromptTemplate.from_template("你是一个专业问答小能手，结合参考资料回答用户问题:\n{content},用户提问:{query}")
prompt=prompt_template.invoke(input={"content":content,'query':query})

text='你是一个专业问答小能手，结合参考资料回答用户问题:\n参考资料: Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###----------------------------\n参考资料:How does Milvus handle vector data types and precision?\n\nMilvus supports Binary, F

In [46]:
print(prompt.to_string())

你是一个专业问答小能手，结合参考资料回答用户问题:
参考资料: Where does Milvus store data?

Milvus deals with two types of data, inserted data and metadata. 

Inserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).

Metadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.

###----------------------------
参考资料:How does Milvus handle vector data types and precision?

Milvus supports Binary, Float32, Float16, a

In [47]:
response=model.invoke(input=prompt)

In [51]:
response.content

'Milvus stores data in two main categories: **inserted data** and **metadata**.\n\n1. **Inserted Data**:  \n   This includes:\n   - **Vector data**: Such as Binary, Float32, Float16, and BFloat16 vectors.\n   - **Scalar data**: Non-vector data associated with the vectors (e.g., text, integers, etc.).\n   - **Collection-specific schema**: The structure or definition of the data stored in a collection.\n\n   Inserted data is stored in **persistent storage** as **incremental logs**. Milvus supports multiple object storage backends for this purpose, including:\n   - MinIO\n   - AWS S3\n   - Google Cloud Storage (GCS)\n   - Azure Blob Storage\n   - Alibaba Cloud OSS\n   - Tencent Cloud Object Storage (COS)\n\n2. **Metadata**:  \n   Metadata is generated and managed within Milvus. Each module of Milvus maintains its own metadata, which is stored in **etcd**, a distributed key-value store.\n\nIn summary, Milvus separates **data** (vectors, scalars, and schemas) from **metadata**, storing them